`py-g5ht-pipeline`

__NOTE__

You will need ~200GB of storage space locally for a single recording. After registration, you can delete ~150GB
- .nd2: 50GB  (~55GB if separate channel alignment recording taken)
- .h5: 9GB
- .tifs created from shear_correction, channel_alignment, bleach_correction: 47GB each     (shear_correction, channel_alignment can be deleted after running the full pipeline)
- .tifs created from warping and registering: 18GB each      (warped can be deleted after running the full pipeline)

## IMPORTS

In [1]:
import sys
import os
import importlib
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import shutil
import utils

## SPECIFY DATA TO PROCESS

In [2]:
data_dir = r'/store1/shared/g5ht/data' # this is a linux machine

# dataset (see datasets.txt)
dataset = 'date-20260112_strain-ISg5HT_condition-fedpatch_worm001.nd2'

NOISE_PTH = '/home/munib/code/g5ht-pipeline/noise/noise_111125.tif'

In [3]:
# directories and paths

INPUT_ND2 = dataset
date_str = INPUT_ND2.split('_')[0].split('-')[1]
local_dir = os.path.join(data_dir, date_str)
os.makedirs(local_dir, exist_ok=True)
INPUT_ND2_PTH = os.path.join(local_dir, INPUT_ND2)


OUT_DIR = utils.get_output_dir(INPUT_ND2_PTH)

STACK_LENGTH = 41 if 'immo' not in INPUT_ND2 else 122

# for recordings prior to roughly December 2025, we want to keep all but the last two z-slices, during which the piezo position is unstable
# after December 2025, we want to keep all but the first two z-slices, during which the piezo position is unstable at the beginning of the recording
date_obj = datetime.strptime(date_str, '%Y%m%d')
if date_obj < datetime(2025, 12, 1):
    z2keep =  (0,STACK_LENGTH-2) # tuple representing range of z-slices to keep, should keep all but the last two slices
else:
    z2keep =  (2,STACK_LENGTH) # tuple representing range of z-slices to keep, should keep all but the first two slices

# get noise stack and metadata from nd2
noise_stack = utils.get_noise_stack(NOISE_PTH, STACK_LENGTH)
num_frames, height, width, num_channels = utils.get_range_from_nd2(INPUT_ND2_PTH, stack_length=STACK_LENGTH) 
beads_alignment_file = utils.get_beads_alignment_file(INPUT_ND2_PTH)

## 1. SHEAR CORRECTION 

` conda activate g5ht-pipeline`

~ 1 hour for 1200 image stacks with 2 color channels, 41 z, 512 h, 512 w

- shear corrects each volume
  - depending on each exposure time, it can take roughly half a second between the first and last frames of a volume, so any movements need to be corrected for
- creates one `.tif` for each volume and stores it in the `shear_corrected` directory

In [4]:
ncpu = utils.get_optimal_cpu_count()

System Load: 3.5% | Available: 61.76 | Allocating: 16


In [ ]:
import shear_correct
_ = importlib.reload(sys.modules['shear_correct'])

start_index = "0"
end_index = str(num_frames-1)
# start_index = "515"
# end_index = str(num_frames-1)

ncpu = str(utils.get_optimal_cpu_count())

skip_shear_correction = False # if True, will just denoise and save as tif

# sys.argv = ["", nd2 file, start_frame, end_frame, noise_pth, stack_length, n_workers, num_frames, height, width, num_channels]
sys.argv = ["", INPUT_ND2_PTH, start_index, end_index, NOISE_PTH, STACK_LENGTH, ncpu, num_frames, height, width, num_channels, z2keep, skip_shear_correction]

# Call the main function
shear_correct.main()